In [ ]:
from torchvision.datasets import MNIST, EMNIST, KMNIST
import torch
from torch import nn
from torchvision import transforms
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F


'\nWe can do:\n- base CIL\n- layer old networks with new ones(meaning the network grows\nand computes a value for how much each independent network\ncontributes to the output, using some kind of final head)\n- CIL/DIL mix, by retraining on old classes while adding new ones(issue: old digits will be higher weighted)\n\n'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN

class NinaProClassificationModel(nn.Module):
  def __init__(self):
    super.__init__()
    self.conv1 = nn.Conv1d(10, 64, kernel_size=5, stride=2)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool1d(pool_size = 2)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool1d(pool_size=2)

    self.flatten = nn.Flatten()
    
    self.dense1 = nn.LazyLinear(256)
    self.relu3 = nn.ReLU()
    self.dense2 = nn.Linear(256, 128)
    self.relu4 = nn.ReLU()
    self.dense3 = nn.Linear(128, 13)

  def forward(self, x):
    out = self.relu1(self.conv1(x))
    out = self.pool1(out)
    out = self.relu2(self.conv2(out))
    out = self.pool2(out)
    out = self.flatten(out)
    out = self.relu3(self.dense1(out))
    out = self.relu4(self.dense2(out))
    out = self.dense3(out)
    return out


# Base NN

transform = transforms.ToTensor()

model = NinaProClassificationModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_batches = 100
batched_train = torch.utils.data.DataLoader(mnist_train, batch_size=num_batches, shuffle=True)
batched_test = torch.utils.data.DataLoader(mnist_test, batch_size=num_batches, shuffle=False)

loss_func = nn.CrossEntropyLoss()

Using cuda device


In [ ]:
print("Starting Training")
model.train()
epochs = 20
for epoch in range(epochs):
  pbar = tqdm(batched_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

  for images, labels in pbar:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    optimizer.zero_grad()
    output = model(images)
    loss = loss_func(output, labels)
    loss.backward()
    optimizer.step()
    pbar.set_postfix(loss=loss.item())

Starting Training


Epoch 9/20:  26%|██▌       | 157/600 [00:02<00:06, 70.17it/s, loss=0.0496]  

In [ ]:
model.eval()
total_loss = 0
total = 0
correct = 0
with torch.no_grad():
  correct = 0
  for images, labels in batched_test:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    logits = model(images)
    loss = loss_func(logits, labels)

    total_loss += loss.item() * images.size(0)
    preds = logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += images.size(0)


print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")

Average Loss: 0.15214015752376192, Accuracy: 0.9748


In [ ]:
# To return to the model

torch.save(model.state_dict(), "mnist_mlp.pt")

In [ ]:
# Re load the pretrained model if needed

model = MLP(layer1, layer2, layer3, 10).to(device)
model.load_state_dict(torch.load("mnist_mlp.pt"))
model.eval()

MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=128, out_features=10, bias=True)
)

In [ ]:
def expand_classifier(model, new_size):
  old_fc = model.fc4
  in_features = old_fc.in_features
  old_out = old_fc.out_features
  newlayer = nn.Linear(in_features, old_out + new_size).to(device)

  with torch.no_grad():
    newlayer.weight[:old_out].copy_(old_fc.weight)
    newlayer.bias[:old_out].copy_(old_fc.bias)
  model.fc4 = newlayer
  return model

In [ ]:
# Base pure CIL

teacher = copy.deepcopy(model).to(device)
for p in teacher.parameters():
    p.requires_grad_(False)
teacher.eval()

Edata_train = EMNIST(root="./data", split="balanced", train=True, download=True, transform=transforms.ToTensor())
letter_indices = torch.where(Edata_train.targets >= 10)[0]
letters_only_train = Subset(Edata_train, letter_indices)

new_train = torch.utils.data.DataLoader(letters_only_train, batch_size=num_batches, shuffle=True)

Edata_test = EMNIST(root="./data", split="balanced", train=False, download=True, transform=transforms.ToTensor())
letter_indices = torch.where(Edata_test.targets >= 10)[0]
letters_only_test = Subset(Edata_test, letter_indices)

new_test = torch.utils.data.DataLoader(letters_only_test, batch_size=num_batches, shuffle=False)
overall_test = torch.utils.data.DataLoader(Edata_test, batch_size=num_batches, shuffle=False)


num_old = 10
num_total = 48
num_newclasses = num_total - num_old

student = expand_classifier(model, num_newclasses)

def calc_accuracies(model, data1, data2, data3):
    model.eval()
    with torch.no_grad():
        correct, total = 0, 0
        for x, y in data1:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
        print("(New Task)Letters acc:", correct / total)
        correct, total = 0, 0
        for x, y in data2:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            # for logit, label in zip(logits, y):
            #   print(f"Logit = {logit.argmax()}, Label = {label}")
            total += y.numel()
        print("(Old Task)Numbers acc:", correct / total)
        correct, total = 0, 0
        for x, y in data3:
            x, y = x.to(device), y.to(device)
            x = torch.flatten(x, start_dim=1)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
        print("(Combined)Total acc:", correct / total)



In [ ]:
# Begin new training
student = student.to(device)
student.train()
optimizer = torch.optim.Adam(student.parameters(), lr=0.001)
epochs = 100

T = 4 # temperature
alpha_kd = .6 # weight of distillation/remembering

for epoch in range(epochs):
  calc_accuracies(student, new_test, batched_test, overall_test)
  pbar = tqdm(new_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
  for images, labels in pbar:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)

    logits_stud = student(images)
    loss_stud_new = F.cross_entropy(logits_stud[:, 10:], labels - 10)

    with torch.no_grad():
      logits_teach = teacher(images)
      loss_teach = F.softmax(logits_teach/T, dim=1)
    loss_stud_old = F.log_softmax(logits_stud[:, :10]/T, dim=1)
    loss_old = F.kl_div(loss_stud_old, loss_teach, reduction="batchmean") * (T**2)

    loss = (1-alpha_kd) * loss_stud_new + alpha_kd * loss_old
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    pbar.set_postfix(loss=loss.item())



(New Task)Letters acc: 0.4027702702702703
(Old Task)Numbers acc: 0.9651
(Combined)Total acc: 0.34164893617021275


(New Task)Letters acc: 0.395
(Old Task)Numbers acc: 0.9646
(Combined)Total acc: 0.3356914893617021


(New Task)Letters acc: 0.3931756756756757
(Old Task)Numbers acc: 0.9642
(Combined)Total acc: 0.33420212765957447


(New Task)Letters acc: 0.40425675675675676
(Old Task)Numbers acc: 0.9675
(Combined)Total acc: 0.3434042553191489


(New Task)Letters acc: 0.41175675675675677
(Old Task)Numbers acc: 0.9646
(Combined)Total acc: 0.34845744680851065


...

(New Task)Letters acc: 0.6818243243243243
(Old Task)Numbers acc: 0.7756
(Combined)Total acc: 0.5586702127659574


(New Task)Letters acc: 0.6864189189189189
(Old Task)Numbers acc: 0.7509
(Combined)Total acc: 0.5621808510638298


Epoch 100/100:  26%|██▌       | 228/888 [00:03<00:09, 71.23it/s, loss=0.0716]

Epoch 100/100:  27%|██▋       | 244/888 [00:03<00:09, 70.96it/s, loss=0.0548]

: 

In [ ]:
torch.save(student.state_dict(), "emnist_mlp.pt")

In [ ]:
student = MLP(layer1, layer2, layer3, 48).to(device)
student.load_state_dict(torch.load("emnist_mlp.pt"))
student.eval()

In [ ]:

calc_accuracies(student, new_test, batched_test, overall_test)

(New Task)Letters acc: 0.5698648648648649
(Old Task)Numbers acc: 0.9218
(Combined)Total acc: 0.47164893617021275


In [ ]:
student.eval()
total_loss = 0
total = 0
correct = 0
with torch.no_grad():
  correct = 0
  for images, labels in batched_test:
    images = images.to(device)
    labels = labels.to(device)
    images = torch.flatten(images, start_dim=1)
    logits = student(images)
    loss = loss_func(logits, labels)

    total_loss += loss.item() * images.size(0)
    preds = logits.argmax(dim=1)
    correct += (preds == labels).sum().item()
    total += images.size(0)

print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")

Average Loss: 0.41778512135589363, Accuracy: 0.9218
